# RQ1 — Per-scheme decomposition (v2 and v3, separately)

Companion to [rq1_directional_reframe.ipynb](rq1_directional_reframe.ipynb)
(pooled, n=76) and [rq1_v2v3_decomposition.ipynb](rq1_v2v3_decomposition.ipynb)
(pooled, two dose-controlled schemes, n=50). This notebook estimates the
geometric effect **separately within each dose-controlled scheme**, so the
result reads as "replicates across two schemes" rather than relying on any
pooled fit. v1/normFalse is excluded (the dose-confounded scheme we debunk
separately).

## Identifiability — why no `mechanical_push` and no scheme dummy

Within a single scheme, `mechanical_push` is a *deterministic* function of
`cos` (per the closed forms in [rq1_consolidated_analysis.ipynb](rq1_consolidated_analysis.ipynb)
§1: `α·√((1+cos)/2)` under normTrue; constant α under per_axis). Including
both `cos` and `mechanical_push` in a within-scheme regression is exact
collinearity — VIF blows up to infinity in the analysis notebook's earlier
audit. Within a scheme we therefore drop `mechanical_push`: the mechanical
dose is already a fixed function of the angle, so `cos` *is* the dose proxy
and the remaining controls only need to absorb amplitude. The scheme dummy is
also dropped because each fit uses a single scheme.

Per-scheme controls: `mean_single_abs + max_single_abs + cos`. Standardized
predictors and outcome (same convention as the prior notebooks).

In [1]:
# Imports + paths + rng + utilities
from pathlib import Path
import json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

REPO = Path.cwd().parents[1]
RNG  = np.random.default_rng(0)

CSV_PATH = REPO / "analysis/rq1_consolidated/consolidated_coh30.csv"
df_all = pd.read_csv(CSV_PATH)
df_all["supp_signed_a"]    = df_all["delta_a_joint"] - df_all["delta_a_single"]
df_all["supp_signed_b"]    = df_all["delta_b_joint"] - df_all["delta_b_single"]
df_all["supp_mean_signed"] = 0.5 * (df_all["supp_signed_a"] + df_all["supp_signed_b"])

df_v2 = df_all[df_all["scheme"] == "normTrue"].reset_index(drop=True)
df_v3 = df_all[df_all["scheme"] == "per_axis"].reset_index(drop=True)

print(f"Full coh≥30 frame: n = {len(df_all)}")
print(f"v2 / normTrue:     n = {len(df_v2)}")
print(f"v3 / per_axis:     n = {len(df_v3)}")
print(f"v2: pairs containing apathetic or hallucinating: "
      f"{((df_v2.trait_a.isin(['apathetic','hallucinating'])) | (df_v2.trait_b.isin(['apathetic','hallucinating']))).sum()}/{len(df_v2)}")
print(f"v3: pairs containing apathetic or hallucinating: "
      f"{((df_v3.trait_a.isin(['apathetic','hallucinating'])) | (df_v3.trait_b.isin(['apathetic','hallucinating']))).sum()}/{len(df_v3)}")

Full coh≥30 frame: n = 76
v2 / normTrue:     n = 28
v3 / per_axis:     n = 22
v2: pairs containing apathetic or hallucinating: 13/28
v3: pairs containing apathetic or hallucinating: 10/22


In [2]:
# Single-scheme OLS — standardize predictors and outcome, no scheme dummy
def fit_ols(X, y):
    X = np.asarray(X, float); y = np.asarray(y, float)
    n, k = X.shape
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    rss = float(resid @ resid)
    dof = n - k
    sigma2 = rss / dof if dof > 0 else np.nan
    XtX_inv = np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    tss = float(((y - y.mean())**2).sum())
    r2 = 1 - rss / tss if tss > 0 else np.nan
    return {"beta": beta, "se": se, "rss": rss, "n": n, "k": k,
            "dof": dof, "r2": r2}


def fit_single_scheme(sub, cols, y_col):
    """Standardized predictors + outcome, intercept only (single scheme)."""
    Z = StandardScaler().fit_transform(sub[cols].values)
    X = np.column_stack([np.ones(len(sub)), Z])
    y = StandardScaler().fit_transform(sub[[y_col]].values).ravel()
    fit = fit_ols(X, y); fit["cols"] = ["(intercept)"] + list(cols)
    return fit


def beta_of(fit, name):
    i = fit["cols"].index(name)
    return float(fit["beta"][i]), float(fit["se"][i])


def beta_cos_single(sub, cols, y_col):
    return beta_of(fit_single_scheme(sub, cols, y_col), "cos")[0]


def perm_p_single(sub, cols, y_col, n_perm=10_000, rng=None):
    rng = rng or RNG
    obs = beta_cos_single(sub, cols, y_col)
    null = np.empty(n_perm)
    sub_loc = sub.copy()
    cos_arr = sub_loc["cos"].values.copy()
    for i in range(n_perm):
        rng.shuffle(cos_arr)
        sub_loc["cos"] = cos_arr
        null[i] = beta_cos_single(sub_loc, cols, y_col)
    return obs, float(np.mean(np.abs(null) >= np.abs(obs)))


def boot_cos_single(sub, cols, y_col, n_boot=5_000, rng=None):
    rng = rng or RNG
    n = len(sub); out = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        out[i] = beta_cos_single(sub.iloc[idx], cols, y_col)
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))


def fit_row(sub, y_col, cols, scheme_label,
            n_perm=10_000, n_boot=5_000):
    fit = fit_single_scheme(sub, cols, y_col)
    b, s = beta_of(fit, "cos")
    t = b / s if s > 0 else np.nan
    p_t = 2 * (1 - stats.t.cdf(abs(t), fit["dof"]))
    r_biv, p_biv = stats.pearsonr(sub["cos"], sub[y_col])
    _, p_perm = perm_p_single(sub, cols, y_col, n_perm=n_perm)
    lo, hi = boot_cos_single(sub, cols, y_col, n_boot=n_boot)
    return {"scheme": scheme_label, "n": fit["n"], "R²": fit["r2"],
            "β_cos (std)": b, "SE": s,
            "p_param": p_t, "perm p": p_perm,
            "boot 95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
            "r(cos, y) biv": r_biv, "p biv": p_biv}


print("Utilities loaded.")

Utilities loaded.


## §1 — v2 / normTrue decomposition

Single-scheme fit: `y ~ mean_single_abs + max_single_abs + cos` (standardized).

In [3]:
COLS = ["mean_single_abs", "max_single_abs", "cos"]

print("=== v2 / normTrue decomposition ===\n")
rows_v2 = [
    fit_row(df_v2, "mean_joint_abs",   COLS, "v2 / normTrue"),
    fit_row(df_v2, "supp_mean_signed", COLS, "v2 / normTrue"),
]
rows_v2[0]["outcome"] = "MAGNITUDE (mean_joint_abs)"
rows_v2[1]["outcome"] = "DIRECTION (supp_mean_signed)"

tbl_v2 = pd.DataFrame(rows_v2)[
    ["outcome","n","R²","β_cos (std)","SE","p_param","perm p","boot 95% CI","r(cos, y) biv","p biv"]
]
print(tbl_v2.round(4).to_string(index=False))

=== v2 / normTrue decomposition ===



                     outcome  n     R²  β_cos (std)     SE  p_param  perm p      boot 95% CI  r(cos, y) biv  p biv
  MAGNITUDE (mean_joint_abs) 28 0.7761       0.1882 0.1241   0.1423  0.0676 [-0.093, +0.415]         0.6459 0.0002
DIRECTION (supp_mean_signed) 28 0.5220       0.3096 0.1813   0.1006  0.0366 [-0.108, +0.637]         0.4754 0.0106


## §2 — v3 / per_axis decomposition

Single-scheme fit: same predictor set.

In [4]:
print("=== v3 / per_axis decomposition ===\n")
rows_v3 = [
    fit_row(df_v3, "mean_joint_abs",   COLS, "v3 / per_axis"),
    fit_row(df_v3, "supp_mean_signed", COLS, "v3 / per_axis"),
]
rows_v3[0]["outcome"] = "MAGNITUDE (mean_joint_abs)"
rows_v3[1]["outcome"] = "DIRECTION (supp_mean_signed)"

tbl_v3 = pd.DataFrame(rows_v3)[
    ["outcome","n","R²","β_cos (std)","SE","p_param","perm p","boot 95% CI","r(cos, y) biv","p biv"]
]
print(tbl_v3.round(4).to_string(index=False))

=== v3 / per_axis decomposition ===



                     outcome  n     R²  β_cos (std)     SE  p_param  perm p      boot 95% CI  r(cos, y) biv  p biv
  MAGNITUDE (mean_joint_abs) 22 0.7457       0.1868 0.1684   0.2819  0.1466 [-0.192, +0.501]         0.6421 0.0013
DIRECTION (supp_mean_signed) 22 0.3492       0.4105 0.2694   0.1449  0.0466 [-0.260, +0.832]         0.4724 0.0264


## §3 — Direction robustness, per scheme

Three probes per scheme on the directional outcome:
- **+ sem_sim** (text-embedding control)
- **|cos| instead of signed**
- **Leave-one-behaviour-out** range across the 8 traits

In [5]:
# 3a. + sem_sim
print("=== 3a. + sem_sim control (directional outcome) ===\n")
for label, sub in [("v2 / normTrue", df_v2), ("v3 / per_axis", df_v3)]:
    cols_sem = ["mean_single_abs", "max_single_abs", "sem_sim", "cos"]
    fit = fit_single_scheme(sub, cols_sem, "supp_mean_signed")
    b, s = beta_of(fit, "cos")
    t = b / s if s > 0 else np.nan
    p = 2 * (1 - stats.t.cdf(abs(t), fit["dof"]))
    _, p_perm = perm_p_single(sub, cols_sem, "supp_mean_signed")
    b_sem, s_sem = beta_of(fit, "sem_sim")
    print(f"  {label}  n={fit['n']}")
    print(f"    β_cos     = {b:+.4f}  SE = {s:.4f}  param p = {p:.4f}  perm p = {p_perm:.4f}")
    print(f"    β_sem_sim = {b_sem:+.4f}  SE = {s_sem:.4f}")
    print()

=== 3a. + sem_sim control (directional outcome) ===



  v2 / normTrue  n=28
    β_cos     = +0.3614  SE = 0.1858  param p = 0.0641  perm p = 0.0170
    β_sem_sim = -0.1667  SE = 0.1464



  v3 / per_axis  n=22
    β_cos     = +0.4433  SE = 0.2943  param p = 0.1503  perm p = 0.0441
    β_sem_sim = -0.0680  SE = 0.2096



In [6]:
# 3b. signed cos vs |cos|, per scheme
print("=== 3b. signed cos vs |cos| (directional outcome) ===\n")
for label, sub in [("v2 / normTrue", df_v2), ("v3 / per_axis", df_v3)]:
    rs, ps = stats.pearsonr(sub["cos"], sub["supp_mean_signed"])
    ra, pa = stats.pearsonr(sub["cos"].abs(), sub["supp_mean_signed"])
    # β_cos with |cos| substituted into the standardized regression
    sub_abs = sub.copy()
    sub_abs["cos"] = sub_abs["cos"].abs()  # substitute |cos| into the cos slot
    fit_abs = fit_single_scheme(sub_abs, COLS, "supp_mean_signed")
    b_abs, s_abs = beta_of(fit_abs, "cos")  # "cos" slot now holds |cos|
    print(f"  {label}  n={len(sub)}")
    print(f"    bivariate r(signed cos, y) = {rs:+.3f}  p = {ps:.4f}")
    print(f"    bivariate r(|cos|,    y)   = {ra:+.3f}  p = {pa:.4f}")
    print(f"    β (|cos|) in partial fit  = {b_abs:+.4f}  SE = {s_abs:.4f}")
    print()

=== 3b. signed cos vs |cos| (directional outcome) ===

  v2 / normTrue  n=28
    bivariate r(signed cos, y) = +0.475  p = 0.0106
    bivariate r(|cos|,    y)   = +0.021  p = 0.9173
    β (|cos|) in partial fit  = +0.0527  SE = 0.1665

  v3 / per_axis  n=22
    bivariate r(signed cos, y) = +0.472  p = 0.0264
    bivariate r(|cos|,    y)   = +0.233  p = 0.2972
    β (|cos|) in partial fit  = +0.1931  SE = 0.2453



In [7]:
# 3c. Leave-one-behaviour-out (directional outcome)
def loo_table(sub, label):
    TRAITS = sorted(set(sub["trait_a"]).union(sub["trait_b"]))
    # Full fit
    b0, s0 = beta_of(fit_single_scheme(sub, COLS, "supp_mean_signed"), "cos")
    rows = [{"drop_trait": "(none — full)", "n": len(sub), "β_cos": b0, "SE": s0}]
    for tr in TRAITS:
        sub_loo = sub[~((sub["trait_a"] == tr) | (sub["trait_b"] == tr))]
        if len(sub_loo) < len(COLS) + 3:
            rows.append({"drop_trait": tr, "n": len(sub_loo), "β_cos": np.nan, "SE": np.nan})
            continue
        b, s = beta_of(fit_single_scheme(sub_loo, COLS, "supp_mean_signed"), "cos")
        rows.append({"drop_trait": tr, "n": len(sub_loo), "β_cos": b, "SE": s})
    loo = pd.DataFrame(rows)
    print(f"=== 3c. LOO ({label}, directional) ===\n")
    print(loo.round(3).to_string(index=False))
    bvals = loo["β_cos"].dropna()
    print(f"\n  β_cos range across drops: [{bvals.min():+.3f}, {bvals.max():+.3f}]")
    print(f"  β_cos full = {b0:+.3f};  median drop = {bvals.iloc[1:].median():+.3f}\n")
    return loo


loo_v2 = loo_table(df_v2, "v2 / normTrue")
loo_v3 = loo_table(df_v3, "v3 / per_axis")

=== 3c. LOO (v2 / normTrue, directional) ===

   drop_trait  n  β_cos    SE
(none — full) 28  0.310 0.181
    apathetic 21  0.064 0.238
   confidence 21  0.405 0.209
         evil 21  0.418 0.178
    formality 21  0.263 0.226
hallucinating 21  0.632 0.239
     humorous 21  0.288 0.196
     impolite 21  0.335 0.240
  sycophantic 21  0.213 0.189

  β_cos range across drops: [+0.064, +0.632]
  β_cos full = +0.310;  median drop = +0.311

=== 3c. LOO (v3 / per_axis, directional) ===

   drop_trait  n  β_cos    SE
(none — full) 22  0.411 0.269
    apathetic 16  0.246 0.441
   confidence 16  0.519 0.291
         evil 16  0.463 0.304
    formality 16  0.384 0.285
hallucinating 18  0.549 0.313
     humorous 19  0.382 0.271
     impolite 16  0.269 0.421
  sycophantic 15  0.430 0.326

  β_cos range across drops: [+0.246, +0.549]
  β_cos full = +0.411;  median drop = +0.407



## §4 — v2 vs v3 side-by-side

Headline comparison printed to stdout so I can paste either column independently
or use both as a "replicates across two schemes" pair.

In [8]:
def compact(r):
    return {"scheme": r["scheme"], "n": r["n"], "β_cos (std)": r["β_cos (std)"],
            "perm p": r["perm p"], "boot 95% CI": r["boot 95% CI"],
            "r(cos, y) biv": r["r(cos, y) biv"], "p biv": r["p biv"]}


print("=" * 72)
print("SIDE BY SIDE — v2 / normTrue vs v3 / per_axis")
print("=" * 72)
print()
print(f"per-scheme n after coh ≥ 30:   v2 = {len(df_v2)},   v3 = {len(df_v3)}")
print()
print("--- MAGNITUDE  (mean_joint_abs) ---")
sxs_mag = pd.DataFrame([compact(rows_v2[0]), compact(rows_v3[0])])
print(sxs_mag.round(4).to_string(index=False))
print()
print("--- DIRECTION  (supp_mean_signed) ---")
sxs_dir = pd.DataFrame([compact(rows_v2[1]), compact(rows_v3[1])])
print(sxs_dir.round(4).to_string(index=False))
print()
print("--- Directional LOO ranges ---")
for label, loo in [("v2 / normTrue", loo_v2), ("v3 / per_axis", loo_v3)]:
    bvals = loo["β_cos"].dropna()
    print(f"  {label:15s}  full β_cos = {bvals.iloc[0]:+.3f}   LOO range [{bvals.min():+.3f}, {bvals.max():+.3f}]   median drop {bvals.iloc[1:].median():+.3f}")
print()
print("=" * 72)

SIDE BY SIDE — v2 / normTrue vs v3 / per_axis

per-scheme n after coh ≥ 30:   v2 = 28,   v3 = 22

--- MAGNITUDE  (mean_joint_abs) ---
       scheme  n  β_cos (std)  perm p      boot 95% CI  r(cos, y) biv  p biv
v2 / normTrue 28       0.1882  0.0676 [-0.093, +0.415]         0.6459 0.0002
v3 / per_axis 22       0.1868  0.1466 [-0.192, +0.501]         0.6421 0.0013

--- DIRECTION  (supp_mean_signed) ---
       scheme  n  β_cos (std)  perm p      boot 95% CI  r(cos, y) biv  p biv
v2 / normTrue 28       0.3096  0.0366 [-0.108, +0.637]         0.4754 0.0106
v3 / per_axis 22       0.4105  0.0466 [-0.260, +0.832]         0.4724 0.0264

--- Directional LOO ranges ---
  v2 / normTrue    full β_cos = +0.310   LOO range [+0.064, +0.632]   median drop +0.311
  v3 / per_axis    full β_cos = +0.411   LOO range [+0.246, +0.549]   median drop +0.407

